In [3]:
# Dash is used to build the webpage and dashboard components
from dash import Dash, dcc, html, dash_table, no_update, ctx
from dash.dependencies import Input, Output, State

# These libraries used for the logo, chart, and map
import base64
import dash_leaflet as dl
import plotly.express as px

# imports on files made for enhancement
from animal_repository import AnimalRepository
from animal_service import AnimalService






# Connection to MongoDB and retrieves records
repository = AnimalRepository()

# handles the rescue filters and prepares the data
service = AnimalService(repository)

# Load all animal records when the dashboard first starts
initial_records = service.get_animals("All")

# Create the column names
table_columns = service.get_table_columns()




app = Dash(__name__)



# Loads logo
image_filename = "Grazioso_Salvare_Logo.png"

# Open the image 
with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(
        image_file.read()
    ).decode()


app.layout = html.Div([

    # Display the Grazioso Salvare logo
    html.Center(
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image),
            style={"height": "200px","width": "200px"})),

    # Display the dashboard title 
    html.Center(
        html.H1("Grazioso Salvare Enhancement")
    ),

    html.Hr(),

    # Create the rescue-type filtering section
    html.Div([

        html.H3("Select a Rescue Category"),

        # These radio buttons allow the user to filter animal records
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "WaterRescue"},
                {"label": "Mountain or Wilderness Rescue", "value": "MWR"},
                {"label": "Disaster or Individual Tracking", "value": "DIT"},
                {"label": "Reset", "value": "All"}
            ],
            value="All",
            inline=True
        ),

        html.Br(),

        # This button allows the user to download the current results
        html.Button(
            "Export Displayed Data",
            id="export-button",
            n_clicks=0
        ),

        # handles the CSV download
        dcc.Download(id="download-csv"),

        #shows how many records were found
        html.Div(id="status-message", children="{} animal records found.".format(
                len(initial_records)
            ),
            style={"marginTop": "10px", "fontWeight": "bold"})]),

    html.Hr(),
    #section to add update and delete animal records
    html.Div([

        html.H3("Database Record Management"),

        html.P("Use the fields below to add, update, or delete an animal record."),

        #animal id
        html.Label("Animal ID"),

        dcc.Input(
            id="input-animal-id",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #animal name
        html.Label("Name"),

        dcc.Input(
            id="input-name",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #animal type
        html.Label("Animal Type"),

        dcc.Input(
            id="input-animal-type",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #animal breed
        html.Label("Breed"),

        dcc.Input(
            id="input-breed",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #animal color
        html.Label("Color"),

        dcc.Input(
            id="input-color",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #sex upon outcome
        html.Label("Sex Upon Outcome"),

        dcc.Input(
            id="input-sex",
            type="text",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        
        html.Label("Age Upon Outcome In Weeks"),

        dcc.Input(
            id="input-age-weeks",
            type="number",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        
        html.Label("Location Latitude"),

        dcc.Input(
            id="input-latitude",
            type="number",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        
        html.Label("Location Longitude"),

        dcc.Input(
            id="input-longitude",
            type="number",
            style={
                "width": "300px"
            }
        ),

        html.Br(),
        html.Br(),

        #button to add animal record
        html.Button(
            "Add Record",
            id="add-button",
            n_clicks=0
        ),

        #button to update animal record
        html.Button(
            "Update Record",
            id="update-button",
            n_clicks=0,
            style={
                "marginLeft": "10px"
            }
        ),

        #button to delete animal record
        html.Button(
            "Delete Record",
            id="delete-button",
            n_clicks=0,
            style={
                "marginLeft": "10px"
            }
        ),

        #message after database action
        html.Div(
            id="crud-message",
            children="Choose an action to manage an animal record.",
            style={
                "marginTop": "10px",
                "fontWeight": "bold"
            }
        ),

        #used to refresh table after database changes
        dcc.Store(
            id="refresh-trigger",
            data=0
        )

    ]),

    html.Hr(),


    # Display the animal records in an interactive table
    dash_table.DataTable(
        id="datatable-id",

        
        columns=table_columns,
        data=initial_records,
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10,

        # Allow horizontal scrolling if the table is too wide
        style_table={
            "overflowX": "auto"
        },

        # Improve the appearance and readability of table cells
        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "120px",
            "maxWidth": "250px",
            "whiteSpace": "normal"
        },

        # Make the table headings easier to identify
        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    #breed chart and location map next to each other
    html.Div(
        style={
            "display": "flex",
            "gap": "20px",
            "flexWrap": "wrap"
        },

        children=[

            #display the breed chart
            html.Div(
                id="graph-id",
                style={
                    "flex": "1",
                    "minWidth": "500px"
                }
            ),

            #animal location map
            html.Div(
                id="map-id",
                style={
                    "flex": "1",
                    "minWidth": "500px"
                }
            )
        ]
    )
])

# Update the table when a filter is selected


@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("status-message", "children")
    ],
    [
        Input("filter-type", "value"),

        #refreshes the table after an add update or delete
        Input("refresh-trigger", "data")
    ]
)
def update_dashboard(filter_type, refresh_value):

    try:
        # search for the matching animal records
        records = service.get_animals(filter_type)

        # Create a message that tells the user how many records were found
        if records:
            message = "{} animal records found.".format(
                len(records)
            )
        else:
            message = ("No animals matched the selected rescue criteria.")

        return records, table_columns, message

    except ValueError:
        #message if an invalid filter is received
        return ([], table_columns, "The selected rescue filter is not valid.")

    except RuntimeError as error:
        # Print error for troubleshooting
        print(error)

        
        return ([], table_columns, "The dashboard could not retrieve animal records.")
#fills the form when a table row is selected


@app.callback(
    [
        Output("input-animal-id", "value"),
        Output("input-name", "value"),
        Output("input-animal-type", "value"),
        Output("input-breed", "value"),
        Output("input-color", "value"),
        Output("input-sex", "value"),
        Output("input-age-weeks", "value"),
        Output("input-latitude", "value"),
        Output("input-longitude", "value")
    ],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def load_selected_record(view_data, selected_rows):

    #does nothing if no row was selected
    if view_data is None or not selected_rows:
        return (
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update
        )

    #gets the selected row
    row_index = selected_rows[0]

    #prevents an invalid row
    if row_index < 0 or row_index >= len(view_data):
        return (
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update,
            no_update
        )

    #gets the animal from the selected row
    animal = view_data[row_index]

    #returns the animal information to the form
    return (
        animal.get("animal_id", ""),
        animal.get("name", ""),
        animal.get("animal_type", ""),
        animal.get("breed", ""),
        animal.get("color", ""),
        animal.get("sex_upon_outcome", ""),
        animal.get("age_upon_outcome_in_weeks", ""),
        animal.get("location_lat", ""),
        animal.get("location_long", "")
    )



#add update or delete database records


@app.callback(
    [
        Output("crud-message", "children"),
        Output("refresh-trigger", "data")
    ],
    [
        Input("add-button", "n_clicks"),
        Input("update-button", "n_clicks"),
        Input("delete-button", "n_clicks")
    ],
    [
        State("refresh-trigger", "data"),
        State("input-animal-id", "value"),
        State("input-name", "value"),
        State("input-animal-type", "value"),
        State("input-breed", "value"),
        State("input-color", "value"),
        State("input-sex", "value"),
        State("input-age-weeks", "value"),
        State("input-latitude", "value"),
        State("input-longitude", "value")
    ],
    prevent_initial_call=True
)
def manage_database_records(
    add_clicks,
    update_clicks,
    delete_clicks,
    refresh_value,
    animal_id,
    name,
    animal_type,
    breed,
    color,
    sex_upon_outcome,
    age_upon_outcome_in_weeks,
    location_lat,
    location_long
):

    #checks which button was clicked
    button_id = ctx.triggered_id

    #stores values entered into the form
    form_data = {
        "animal_id": animal_id,
        "name": name,
        "animal_type": animal_type,
        "breed": breed,
        "color": color,
        "sex_upon_outcome": sex_upon_outcome,
        "age_upon_outcome_in_weeks": age_upon_outcome_in_weeks,
        "location_lat": location_lat,
        "location_long": location_long
    }

    try:
        #adds an animal record
        if button_id == "add-button":
            service.create_animal(form_data)

            return (
                "Animal record was added successfully.",
                refresh_value + 1
            )

        #updates an animal record
        if button_id == "update-button":
            updated_count = service.update_animal(
                animal_id,
                form_data
            )

            return (
                "{} animal record(s) were updated successfully.".format(
                    updated_count
                ),
                refresh_value + 1
            )

        #deletes an animal record
        if button_id == "delete-button":
            deleted_count = service.delete_animal(
                animal_id
            )

            return ("{} animal record(s) were deleted successfully.".format(deleted_count),
                refresh_value + 1
            )

        return no_update, refresh_value

    except ValueError as error:
        #shows input validation error
        return str(error), refresh_value

    except RuntimeError as error:
        #prints database error
        print(error)

        return ("The database action could not be completed.", refresh_value)
# Update the chart using the visible records


@app.callback(
    Output("graph-id", "children"),
    Input("filter-type", "value")
)
def update_graph(filter_type):
  

    #count the most common breeds
    summary = service.get_database_breed_summary(filter_type)

    #shows message if chart is empty
    if summary.empty:
        return html.P("No breed information is available.")

    # Create a bar chart showing the ten most common breeds
    figure = px.bar(summary,x="breed",y="count",title="Top Breeds in Current Results",
        labels={"breed": "Breed","count": "Number of Animals"})

    # Angled names so you can actually read them
    figure.update_layout(xaxis_tickangle=-35)

    return dcc.Graph(figure=figure)



# Highlight a selected table column


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id","selected_columns"))
def update_styles(selected_columns):
    #Highlight the column selected

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "#F2F2F2" #light gray
        }
        for column in selected_columns
    ]



# Update the map using the selected animal


@app.callback(
    Output("map-id", "children"),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(view_data, selected_rows):
    #Displays the location of the selected animal

    # Use the starting records if the table data is not available
    if view_data is None:
        view_data = initial_records

    
    animal = service.get_selected_animal(
        view_data,
        selected_rows
    )

    # Display a message if the animal does not have a valid location
    if animal is None:
        return html.P("No location is available for this animal.")

    # Create a map centered on the selected animal
    return dl.Map(
        style={"width": "100%","height": "500px"},

        center=[
            animal["latitude"],
            animal["longitude"]
        ],

        zoom=10,

        children=[

            
            dl.TileLayer(),

            # Places a marker for location
            dl.Marker(
                position=[
                    animal["latitude"],
                    animal["longitude"]
                ],

                children=[

                    
                    dl.Tooltip(
                        animal["breed"]
                    ),

                    
                    dl.Popup([

                        html.H3(
                            "Animal Information"
                        ),

                        html.P(
                            "Name: {}".format(
                                animal["name"]
                            )
                        ),

                        html.P(
                            "Breed: {}".format(
                                animal["breed"]
                            )
                        )
                    ])
                ]
            )
        ]
    )



# Export to a CSV file


@app.callback(
    Output("download-csv", "data"),
    Input("export-button", "n_clicks"),
    State("datatable-id", "derived_virtual_data"),
    prevent_initial_call=True
)
def export_results(n_clicks, view_data):
    #Download the records displayed in the table

    if not n_clicks:
        return no_update

    # Use all starting records if the table wasnt changed
    if view_data is None:
        view_data = initial_records

    # doesnt create empty file
    if not view_data:
        return no_update

    try:
        csv_content = service.records_to_csv(
            view_data
        )

        return {"content": csv_content, "filename": "animal_shelter_results.csv", "type": "text/csv"}

    except ValueError:
        return no_update






app.run(
    debug=False,
    port=8050,
    jupyter_mode="tab" #opens in a tab instead of cell
)

Connected to MongoDB
Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>